## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...
            </span>
        </td>
    </tr>
</table>

In [1]:
# Start with imports - ask ChatGPT to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [2]:
# Always remember to do this!
load_dotenv(override=True)

True

In [3]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI
DeepSeek API Key exists and begins sk-
Groq API Key exists and begins gsk_


In [4]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [4]:
messages

[{'role': 'user',
  'content': 'Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. Answer only with the question, no explanation.'}]

In [5]:
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=messages,
)
question = response.choices[0].message.content
print(question)


You are responsible for designing an automated decision-making system (ADS) to allocate a limited fleet of emergency medical helicopters in a mid-sized city that has significant demographic disparities and a risk of adversarial actors who may place false emergency calls to monopolize resources. Raw personal medical records are off-limits; you only have access to noisy real-time sensor feeds, call metadata, and historical anonymized aggregated outcomes. Your objectives, in descending priority, are: (1) maximize lives saved (measured as expected reduction in mortality versus the current baseline), (2) minimize time-to-treatment for high-risk patients, (3) ensure equitable service across demographic groups, and (4) be robust to gaming and sensor failure—while operating under a fixed budget and legal/explainability constraints for public oversight. Describe the following: (A) a precise formal objective function (or multi-objective formulation) and decision variables that encode these prior

In [6]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

## Note - update since the videos

I've updated the model names to use the latest models below, like GPT 5 and Claude Sonnet 4.5. It's worth noting that these models can be quite slow - like 1-2 minutes - but they do a great job! Feel free to switch them for faster models if you'd prefer, like the ones I use in the video.

In [7]:
# The API we know well
# I've updated this with the latest model, but it can take some time because it likes to think!
# Replace the model with gpt-4.1-mini if you'd prefer not to wait 1-2 mins

model_name = "gpt-5-nano"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

Below is a concrete design sketch for an automated decision-making system (ADS) to dispatch a fixed fleet of emergency medical helicopters under the constraints you stated. I state key assumptions explicitly, then present (A)–(F) as requested, with trade-offs justified.

Overview and core philosophy
- We operate in a real-time, noisy-data setting with adversarial risk (false emergency calls) and significant demographic disparities. Raw PII is off-limits; we rely on noisy sensor streams, call metadata, and aggregated historical outcomes.
- The objective is multi-objective and risk-aware: maximize expected lives saved (primary), minimize time-to-treatment for high-risk patients, ensure equitable access across demographic groups, and be robust to adversarial gaming and sensor failures, all within a fixed budget and with public explainability requirements.
- We implement a rolling-horizon, robust, multi-objective dispatch policy with a human-in-the-loop fallback when necessary. We separate estimation (demand, probabilities of real vs fake) from optimization (assignment of helicopters) and maintain privacy by design (aggregate features, differential privacy where feasible).

A. Precise objective function and decision variables (multi-objective formulation)
Assumptions you should confirm or adjust:
- There are H helicopters in service; each helicopter can serve at most one active incident at a time.
- Time is discretized into decision epochs t ∈ {0,1,…,T−1} (e.g., every 30 seconds to 2 minutes). We solve a rolling optimization at each epoch.
- Incidents arrive as a stream I_t. Each incident i ∈ I_t has:
  - location l_i, current time t_i, a severity proxy s_i ∈ {1,…,S}, associated base mortality risk if untreated, and a demographic group g(i) ∈ G (where G is the set of demographic groups considered for equity).
  - call metadata and sensor-derived indicators that feed a probabilistic model of realism: p_i(real) = p_i, and we also compute p_i^LB (a conservative lower bound on p_i(real)) to capture uncertainty (adversaries may try to game the system).
- For tractability, we treat each decision epoch as selecting a matching between helicopters and a subset of currently active incidents; unserved incidents remain on the queue to be considered in next epochs.

Decision variables
- a_{h,i} ∈ {0,1} for each helicopter h ∈ H and incident i ∈ I_t: 1 if helicopter h is assigned to incident i in the current horizon; 0 otherwise.
- t_i ∈ ℝ: predicted arrival/dispatch time for incident i if assigned this epoch (derived from current_time, path travel time, and service time). In MILP form this can be linearized with big-M terms.
- b_h ∈ {0,1}: helicopter h is positioned/holding (idle) rather than dispatched in this epoch (optional, used for repositioning strategies).

Derived/auxiliary quantities
- Travel_time(h,i) = time to reach i from current base/position of helicopter h (could include re-positioning time to a staging location if used).
- Service_time_i = time to prepare and begin treatment at the scene (or time to complete stabilization).
- Arrival_time_i = current_time + Travel_time(h,i) when i is assigned to h.
- Time_to_treatment_i = Arrival_time_i − current_time (or actual treatment start time relative to incident time), used to model mortality reduction as a function of response time.

Probability-weighted value for each potential assignment
- Realism p_i^LB ∈ [0,1]: a conservative lower bound on the probability that incident i is a genuine emergency (not a gaming attempt). This is computed from the metadata plus sensor indicators with privacy-preserving features.
- Mortality reduction function M_i(T) captures the expected lives saved if incident i is treated with arrival time T. A simple parametric form (to be calibrated offline) is:
  M_i(T) = M_i,max · exp(−η_i · (T − T0_i)) for T ≥ T0_i, and M_i(T) ≡ M_i,max for T ≤ T0_i,
  where M_i,max is the maximum possible mortality reduction for incident i, η_i > 0 is a severity- and context-dependent decay rate, and T0_i is a gold-time threshold for i.
- A fairness (equity) term: for each demographic group g ∈ G, define P_served(g) as the share of served incidents originating from group g in the horizon, and P_target(g) as the policy target share (or a target range). The fairness penalty can be:
  Fairness = ∑_{g∈G} w_g · (P_served(g) − P_target(g))^2,
  with weights w_g reflecting policy emphasis on groups.

Multi-objective scalarization (practical approach)
- We use a scalarized objective with explicit weights, allowing the city to tune emphasis. A representative form is:
Maximize
  V = ∑_{i∈I_t} ∑_{h∈H} a_{h,i} · p_i^LB · [β1 · M_i(Arrival_time_i) − β2 · Arrival_time_i]
      − β3 · Fairness
+ or − additional terms for robustness and operational constraints.

Where β1, β2, β3 ≥ 0 are trade-off weights chosen by policy:
- β1 prioritizes expected lives saved (via p_i^LB and the mortality-reduction function).
- β2 penalizes late arrivals to incentivize faster response.
- β3 penalizes inequity between demographic groups.

Constraints (key ones)
- Each helicopter handles at most one incident per decision epoch: ∑_{i∈I_t} a_{h,i} ≤ 1 for all h ∈ H.
- Each incident is served by at most one helicopter: ∑_{h∈H} a_{h,i} ≤ 1 for all i ∈ I_t.
- Time feasibility: If a_{h,i} = 1, then Arrival_time_i ≤ t_deadline_i; otherwise i is not selected this epoch (or can be carried to future).
- Re-positioning constraint (optional): If b_h = 1, helicopter h cannot serve an incident in this epoch; used to optimize geography, minimize cross-coverage gaps, etc.
- Equity constraint (optional hard or soft): enforce bounds on how many incidents from each group can be served in this epoch, or embed in the fairness penalty as above.

Adversarial robustness modeling
- Realism probabilities p_i^LB serve as a robustness hedge: even if a fraction of incoming calls are adversarial (p_i(real) low), we optimize using the conservative bound p_i^LB.
- Optional augmentation: a constraint that caps the number of served calls from any small geographic area or any time window to deter gaming by clustering fake calls—a locality-based robustness guardrail.
- If you want an explicit game-theoretic flavor, you can recast as a two-player zero-sum game (Defender: dispatcher; Attacker: initiator of false calls) and optimize a robust surrogate (e.g., a min-max problem with an uncertainty set for p_i(real)); in practice we implement this by using p_i^LB and a separate risk-limiting constraint.

Explicit state/decision notation recap
- H: set of helicopters
- I_t: active incident set at epoch t
- a_{h,i} ∈ {0,1}: helicopter i assigned to incident i
- Arrival_time_i = current_time + Travel_time(h,i) for chosen h with a_{h,i} = 1
- p_i^LB: conservative lower bound probability incident i is real
- M_i(T): mortality reduction for incident i given arrival time T
- Fairness term as defined above
- β1, β2, β3: policy weights to balance lives saved, time-to-treatment, and fairness

Trade-offs justified in (A)
- Lives saved vs time-to-treatment: increasing β1 relative to β2 makes the optimizer favor assignments that maximize expected mortality reduction, potentially by selecting faster responses for high-severity incidents. Conversely, increasing β2 reduces the emphasis on ultra-fast arrivals if the marginal mortality benefit is small.
- Fairness vs efficiency: increasing β3 enforces equity across groups but may reduce total lives saved or increase average treatment times if non-uniform demand concentrates in under-served groups. The policy should reflect public accountability: start with modest equity weighting and increase if disparity metrics exceed targets.
- Robustness vs perfomance: using p_i^LB as the realism bound improves protection against adversaries but may under-serve legitimate incidents if the bound is overly conservative. Calibrate p_i^LB using historical false-call rates and current sensor reliability; consider adaptive reweighting as data drift is detected.
- Explainability: a single scalar objective with interpretable components (M_i, arrival times, group shares) supports public explainability. If the city requires human-readable rationales, we can provide per-dispatch justifications grounded in the features (e.g., “high-severity incident with rapid mortality risk reduction and no demonstrated gaming signal”).

B. Algorithmic approach for real-time dispatch (uncertainty, adversaries) and deployment considerations
Assumptions
- The city uses a rolling horizon (e.g., 60–180 seconds), updating decisions at a fixed cadence (every 15–30 seconds to maintain responsiveness).
- The problem size per epoch is moderate (e.g., up to ~K helicopters and a few dozen active incidents in a mid-sized city).
- Data latency is nonzero and sensor streams can sometimes drop; we must be robust to missing data.

Proposed approach
1) Demand and realism estimation (pre-dispatch module)
- Build a lightweight probabilistic model (e.g., logistic regression, gradient-boosted trees, or a small neural net) that ingests:
  - Call metadata: time of day, location, caller type, speed/acceleration of vehicle, call duration, pattern features.
  - Sensor hints (from on-scene or vehicle sensors) such as approximate vitals or scene indicators when available, plus environment features (weather, traffic).
- Output for each incident i: p_i^LB, a realism score s_i (optional), and a short quality indicator q_i (confidence in the estimate).
- Privacy-preserving: operate on de-identified and aggregated features; if needed, apply differential privacy to share counts or to publish aggregated statistics.

2) Rolling-horizon robust optimization (core dispatcher)
- Solve the MILP (as in A) at each epoch with current I_t, current helicopter positions, Travel_time(h,i) updated, and p_i^LB inserted in the objective.
- Use a two-stage approach for latency:
  - Stage 1: solve a compact MILP with current active incidents (I_t) and a subset of the most promising frictions (high-severity or high-p_i^LB) to obtain a near-optimal assignment quickly.
  - Stage 2: refine by adding nearby incidents or re-optimizing with updated data if time permits (warm starts from Stage 1).
- Use warm-starts: reuse the previous epoch’s assignment as a starting point to speed convergence.
- Heuristic fallback: if MILP runtime exceeds a bound (e.g., 2–3 seconds), switch to a fast greedy heuristic:
  - Rank pairs (h,i) by a score S(h,i) = p_i^LB · (β1 · M_i(Arrival_time_i) − β2 · Arrival_time_i) − fairness_penalty_component, and assign iteratively while respecting constraints.
- Adversarial robustness: p_i^LB is used to dampen the impact of calls that may be adversarial; include optional hard constraints to cap the number of incidents served from a single geographic grid cell per epoch to deter gaming.

Complexity, latency, and deployment considerations
- MILP size: with H helicopters and N active incidents, there are O(HN) binary variables and constraints. For H in the 4–8 range and N in the 20–60 range, modern MILP solvers typically finish in sub-2–5 seconds on a mid-range server; this fits a rolling window of 15–30 seconds in practice.
- Real-time requirements: ensure worst-case solve time is bounded by a fail-safe (e.g., if the MILP fails to return within 3–4 seconds, fall back to the heuristic and log for offline analysis).
- Data locality and privacy: run optimization in a secured dispatch center with access to anonymized, aggregated features; keep PII out of the optimization core.
- Deployment: pilot in shadow or soft-launch mode (see D). Start with a human-in-the-loop baseline and gradually increase autonomy.

D. Data pipeline, privacy-preserving methods, and minimal needed features
Assumptions
- Data sources include: real-time helicopter sensor feeds, EMS call metadata, traffic/weather data, hospital status dashboards, and historical anonymized outcomes.
- PII cannot be used in the optimization; only aggregated or de-identified signals are allowed in live operation.

Pipeline design
1) Ingestion
- Streaming system (e.g., Apache Kafka) to accept calls, sensor feeds, weather, and hospital statuses.
- Data quality checks: timestamp alignment, missing-value flags, sensor health indicators.

2) De-identification and privacy
- Redact direct identifiers; replace with pseudo IDs; aggregate counts by area rather than tracking individuals.
- Apply differential privacy to population-level statistics used for fairness metrics or regional demand estimates when sharing externally; keep internal models using non-private representations when possible.
- Role-based access control and audit trails for all data access.

3) Feature engineering (minimal, privacy-preserving)
- Incident-level features (derived, not raw): 
  - Location grid cell g(i), proximity to nearest base/hospital, time since incident start, weather-adjusted travel times.
  - Severity proxy s_i, high-level sensor anomaly score a_i (0..1), realism proxy p_i^LB.
  - Group indicator g(i) (e.g., census block group or coarse demographic label tied to public data).
- System-level features:
  - Helicopter statuses: current base locations, fuel state, maintenance windows, and time-to-deploy for repositioning.
  - Regional demand indicators: recent incident counts per grid cell, seasonal effects.
- Derived metrics (privacy-safe) for equity: counts of incidents per group by area, share per group, baseline and target shares.

4) Privacy-preserving analytics
- Use DP-safe aggregations for any group-level metrics that inform fairness constraints.
- Secure multi-party-like considerations only if multiple agencies feed data; otherwise keep centralized.
- Data retention: keep raw streams only for a bounded window (e.g., 30–60 days) for model retraining; retain only anonymized aggregates for longer-term dashboards.

5) Model update and governance
- Periodic retraining with anonymized historical data; monitor drift in p_i^LB calibration and M_i(T) calibration (see E).
- Explainability: maintain logs of major features driving each dispatch decision (without exposing PII) to support public reporting and oversight.

Minimal features necessary (live)
- incident i: l_i, t_i, s_i, p_i^LB, a_i (anomaly flag), g(i), Travel_time(h,i) for nearby bases (computed from static maps and current traffic).
- helicopter h: current position, time to ready/launch, base, fuel state.
- system-wide: current epoch time, weather status, hospital capacity snapshot.

E. Evaluation, validation, and continuous monitoring plan (pre- and post-deploy)
Assumptions
- A baseline dispatch protocol exists (e.g., nearest-available or base-first approach) to compare against.

Pre-deployment (offline simulation and validation)
- Data: historical anonymized call metadata, sensor proxies, and aggregated outcome data; synthetic adversarial scenarios generated by perturbing a fraction of calls to simulate false alarms.
- Methods:
  - Backtesting across multiple time windows and neighborhoods.
  - Cross-validation across different city regions (to test generalizability).
  - Simulation of adversarial load: inject up to a fraction A of calls with low p_i(real) and measure impact.
- Metrics (with targets):
  - LivesSaved_diff = expected lives saved by ADS vs baseline (positive target; e.g., 5–15% improvement depending on city size).
  - Time_to_treatment_mean and 95th percentile (lower is better; compare to baseline).
  - Equity_disparity = max_g P_served(g) − min_g P_served(g); target a small value (e.g., ≤ 0.05 in share terms) or a quadratic penalty below a threshold.
  - Adversarial robustness: change in LivesSaved under injected adversarial demand; require less than a specified drop (e.g., < 2–5% under simulated worst-case load).
  - Fairness tests: permutation tests to ensure observed disparities are not statistically significant at α = 0.05 after accounting for demand.
  - Explainability: provide per-dispatch rationales and summary feature importance histograms; target a threshold for average number of features needed to explain decisions (e.g., top-3 features suffice 80% of the time).

Post-deployment (live monitoring and adjustment)
- Metrics and monitoring:
  - Real-time LivesSaved_estimate vs baseline; monitor confidence intervals.
  - Time-to-treatment distribution; SLA adherence rates.
  - Equity indicators: rolling window disparities by group; alert if exceeding target bounds.
  - Robustness indicators: detection rate of anomalous calls; false-positive rate of the realism classifier; number of escalations to human dispatch.
  - System reliability: latency distribution of MILP solve, fallback frequency, and dispatcher queue growth.
  - Safety and explainability: number of decisions with low explainability scores, per-dispatch rationales generated, and oversight reviews.
- Statistical tests and thresholds:
  - Weekly paired t-tests or bootstrap tests comparing ADS vs baseline on key metrics if a historical-control period is available.
  In the absence of a perfect control, use synthetic control methods or A/B testing with a shadow mode (see below).
- Rollout strategy:
  - Shadow mode: live system produces dispatch recommendations and feeds results back to a separate evaluator stream; no real resource allocation changes. Measures alignment with actual outcomes to build trust and calibrate safety margins.
  - Pilot with human-in-the-loop: gradually allow the ADS to authorize dispatch with a supervisor override option; measure how often supervisor changes decisions.
- Continuous improvement:
  - Regular retraining cadence (e.g., monthly) with updated data; monitor drift in p_i^LB calibration, M_i(T) calibration, and fairness metrics.
  - Update risk weights (β1, β2, β3) if real-world results differ from expectations, with governance approvals.

Fairness and safety governance
- Predefine target equity goals (P_target(g)) and acceptable disparity thresholds; adjust weights to enforce or relax as policy changes.
- Maintain a public explainability document detailing the decision logic and key features used per decision, without exposing sensitive patient data.
- Maintain a kill-switch and manual override to revert to the baseline dispatch protocol within seconds if a safety issue is detected.

F. Minimal clarifying questions to city officials (underspecification resolved)
- Fleet and geography
  - How many helicopters are funded and available daily? Are there fixed bases or can helicopters reposition?
  - Are there time-of-day or day-of-week constraints (e.g., night operations, maintenance windows)?
  - Are repositioning/redeployment policies allowed (e.g., moving a helicopter closer to high-demand zones during peak hours)?
- Performance targets and fairness
  - What are the official performance targets for response times (e.g., median response within X minutes, 90th percentile within Y minutes)?
  - How is equity defined in policy terms? Which demographic groups should receive protection, and what are the acceptable disparity thresholds?
  - Do you require strict hard constraints on service to certain areas, or is it a soft objective with penalties?
- Data and privacy
  - Which data sources are available for real-time use, and which are permissible for live optimization?
  - Are there constraints on sharing or aggregating data with the public or with hospitals? Any restrictions on differential privacy levels?
  - What privacy controls must be in place for data used in the live decision loop?
- Oversight, explainability, and auditing
  - What level of explainability is required for public oversight (e.g., per-dispatch rationale, feature importances, or a summarized policy narrative)?
  - How often should audit logs be produced, and who has access to them?
  - Is there a legal or policy requirement to publish annual fairness/equity reports?
- Adversarial risk tolerance
  - What is the acceptable level of risk from adversarial calls? Is there a formal adversarial risk assessment or tabletop exercise plan?
  - Are there constraints on implementing guardrails (e.g., hard caps on number of served calls per area per hour)?
- Operational integration
  - What is the preferred rollout pathway (shadow mode → controlled pilot → full deployment)?
  - What is the acceptable fallback mode if the ADS fails (e.g., revert to nearest-available rule, or maintain current manual dispatch protocols with decision logs)?
- Evaluation and accountability
  - What metrics are publicly reportable? What thresholds constitute success for the first year?
  - Who is responsible for sign-off on model updates and changes to weights or policy targets?

Key assumptions recap (to align with your request)
- PII safety: We assume raw medical records are unavailable and cannot be used; all live optimization uses de-identified or aggregated features, with privacy protections (DP where feasible).
- Data quality: Sensor feeds can be noisy and intermittently unavailable; the system tolerates missing data via robust estimation and conservative realism probabilities.
- Adversary model: We do not assume perfect knowledge of adversaries; we handle gaming risk by using conservative realism bounds (p_i^LB) and by limiting resource concentration per grid/time window as guardrails.
- Explainability: Public oversight requires transparent rationale; the model provides disclosure-friendly per-dispatch rationales and aggregated feature importance.
- Deployment risk: Start with shadow or human-in-the-loop pilot; gradually scale to autonomous dispatch with appropriate supervision.

Main trade-offs summarized
- Lives saved vs fairness: A heavier emphasis on mortality reduction generally improves aggregate lives saved but may create disparities if demand concentrates in underprivileged areas; equity constraints and/or penalties can mitigate this at the cost of some lost lives.
- Robustness vs efficiency: Conservative realism (p_i^LB) reduces risk from gaming but may under-allocate to some real incidents. Calibrating p_i^LB and guardrails helps strike a balance; more aggressive guardrails may reduce overall throughput.
- Explainability vs optimization complexity: A transparent scalarized objective with explicit components supports oversight but can be harder to capture nuanced trade-offs; a well-chosen set of features and per-dispatch rationales helps maintain clarity without sacrificing too much optimization performance.
- Privacy vs data utility: Strict privacy constraints limit the granularity of optimization data. Using aggregated, minimally sufficient features preserves privacy while still enabling effective dispatch decisions.

If you’d like, I can tailor the A–F sections to a specific city profile (e.g., helicopter fleet size, typical demand patterns, available data streams) and provide a concrete MILP formulation (with variable indices, constraints, and a small example) you can hand to a pilot optimizer to prototype.

In [8]:
# Anthropic has a slightly different API, and Max Tokens is required

model_name = "claude-sonnet-4-5"

claude = Anthropic()
response = claude.messages.create(model=model_name, messages=messages, max_tokens=1000)
answer = response.content[0].text

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

# Comprehensive Design for Emergency Medical Helicopter Allocation System

## A. Formal Objective Function and Decision Variables

### Decision Variables

**Primary variables:**
- `x_{i,t} ∈ {0,1}`: Binary assignment of helicopter `i` to call `t` at time step `τ`
- `r_{i,τ}`: Routing decision for helicopter `i` at time `τ` (location vector)
- `p_t ∈ [0,1]`: Estimated legitimacy probability for call `t`

**State variables:**
- `s_{i,τ}`: Current state of helicopter `i` (available, en-route, on-scene, maintenance)
- `q_τ`: Queue of pending calls at time `τ`

### Multi-Objective Formulation

```
Primary Objective (Lexicographic ordering with constraints):

L1: max Σ_t [p_t · α_t · I(response_time_t ≤ golden_hour) · x_{*,t}]

Subject to:
L2: E[response_time_t | severity_t = HIGH, x_{*,t} = 1] ≤ β
L3: |service_rate_g - service_rate_overall| ≤ ε ∀ demographic_group g
L4: robustness_score ≥ γ

Where:
- α_t: Expected lives saved coefficient (severity-weighted)
- p_t: Legitimacy probability (adversarial filtering)
- golden_hour: Critical time window (parameterized by condition)
- β: Time threshold for high-risk patients
- ε: Fairness tolerance bound
- γ: Minimum robustness threshold
```

### Detailed Components

**Lives Saved (L1):**
```
α_t = f(severity_t, comorbidity_proxy_t, distance_to_facility_t)

severity_t = g(call_metadata, sensor_correlates, historical_patterns)

Expected mortality reduction:
ΔM_t = baseline_mortality_t - intervention_mortality_t(response_time)
```

**Fairness Constraint (L3):**
```
service_rate_g = (calls_served_g / calls_eligible_g) for group g

Measured across:
- Geographic zones (proxy for socioeconomic status)
- Time-of-day patterns (work vs. residential areas)
- Historical service coverage gaps

Constraint type: Disparate impact minimization with tolerance ε = 0.15
(allows 15% deviation, justified by statistical noise in small samples)
```

**Robustness Score (L4):**
```
robustness_score = w_1·adversarial_detection_precision 
                  + w_2·sensor_failure_recovery_rate
                  + w_3·prediction_calibration_score

With w_1=0.5, w_2=0.3, w_3=0.2 (adversarial resistance prioritized)
```

### Key Assumptions & Trade-offs

**Assumptions:**
1. Historical anonymized data is representative of future patterns (stationarity assumption with drift detection)
2. Legitimacy probability can be estimated from metadata patterns without protected attributes
3. Geographic zones serve as reasonable proxies for demographic groups (Census-tract level)
4. Lives saved can be approximated through severity-response time curves from trauma literature

**Trade-offs:**
- **Lives vs. Fairness**: Lexicographic ordering ensures we don't sacrifice more than ε fairness for marginal life-saving gains. This is ethically justified: preventing systematic discrimination is a hard constraint, not a soft preference.
- **Optimality vs. Explainability**: Using rule-based severity tiers (Level 1-5) with transparent weights rather than pure deep learning for α_t estimation. Accepts ~5-10% optimality loss for interpretability.
- **Precision vs. Recall (adversarial detection)**: Set p_t threshold at 0.3 (low bar) to favor false positives over false negatives—better to respond to some fake calls than miss real emergencies. Costs ~10-15% resource waste but prevents catastrophic failures.

---

## B. Algorithmic

In [9]:
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.5-flash"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

This response outlines the design of an Automated Decision-Making System (ADS) for emergency medical helicopter allocation, addressing the specified requirements and constraints.

---

### A. Precise Formal Objective Function and Decision Variables

**Assumptions:**
*   We have a probabilistic model, $P(\text{survival} | \text{incident_type}, \text{patient_features}, \text{treatment_time})$, which estimates the likelihood of survival given the incident characteristics, inferred patient risk factors (from call metadata/sensor data, *not* PII), and the time until treatment. This model is trained on historical anonymized aggregated outcomes.
*   "High-risk patients" are identified through a classification model based on call metadata and sensor data, assigning a risk score $R_p \in [0,1]$.
*   Demographic groups $G = \{g_1, g_2, ..., g_k\}$ are defined based on anonymized, aggregated, and legally compliant categories (e.g., census tracts, income quartiles, age brackets) rather than individual PII.
*   A mechanism exists to assign a confidence score $C \in [0,1]$ to each emergency call, indicating its likelihood of being genuine versus adversarial.
*   The system operates in discrete time steps $\Delta t$.

**Decision Variables:**
For each incoming emergency call $i$ at time $t$ and for each available helicopter $h$:
*   $x_{i,h,t} \in \{0, 1\}$: Binary variable. 1 if helicopter $h$ is dispatched to call $i$ at time $t$, 0 otherwise.
*   $y_{i,t} \in \{0, 1\}$: Binary variable. 1 if call $i$ is confirmed as valid and requiring a helicopter at time $t$, 0 if it's held for verification or classified as false.

**Parameters:**
*   $S_i(T)$: Expected survival probability for incident $i$ if treatment starts at time $T$.
*   $T_h(loc_i, loc_h)$: Travel time for helicopter $h$ from its current location $loc_h$ to incident $loc_i$.
*   $T_{current}$: Current time.
*   $W_{risk}$: Weight for high-risk patients (e.g., 1.5 for high-risk, 1 for others).
*   $D_{g}$: Demographic group of incident $i$.
*   $\bar{E}_{g}$: Target equitable service metric for group $g$ (e.g., average time-to-treatment, proportion of calls dispatched).
*   $E_{g,t}$: Current observed service metric for group $g$ up to time $t$.
*   $N_{heli}$: Total number of available helicopters.
*   $K_{max}$: Maximum acceptable number of unverified dispatches (for robustness).
*   $\lambda_1, \lambda_2, \lambda_3$: Positive weighting factors for lives saved, high-risk patient time-to-treatment, and equity, respectively. $\lambda_1 \gg \lambda_2 > \lambda_3$ reflects priority order.
*   $\alpha$: Penalty coefficient for dispatching to unconfirmed calls, increasing with confidence $C$.

**Multi-Objective Formulation:**

The primary objective is to maximize the expected lives saved. We incorporate the other objectives as weighted penalty terms or constraints.

**Objective Function:**
$$ \text{Maximize } \quad J = \sum_{i \in \text{incoming calls}} \sum_{h \in \text{available heli}} \left[ \lambda_1 \cdot W_{risk,i} \cdot S_i(T_{current} + T_h(loc_i, loc_h)) \cdot x_{i,h,t} \cdot y_{i,t} \right] $$
$$ - \lambda_2 \cdot \sum_{i \in \text{high-risk calls}} \sum_{h \in \text{available heli}} \left[ \max(0, (T_{current} + T_h(loc_i, loc_h)) - T_{target,i}) \cdot x_{i,h,t} \cdot y_{i,t} \right] $$
$$ - \lambda_3 \cdot \sum_{g \in G} \left[ \text{FairnessPenalty}(E_{g,t}, \bar{E}_{g}) \right] $$
$$ - \alpha \cdot \sum_{i \in \text{incoming calls}} \sum_{h \in \text{available heli}} \left[ (1 - C_i) \cdot x_{i,h,t} \cdot (1 - y_{i,t}) \right] $$

Where:
*   $W_{risk,i}$: A weight assigned to incident $i$. If incident $i$ is classified as high-risk, $W_{risk,i} > 1$. Otherwise, $W_{risk,i} = 1$. This directly amplifies the "lives saved" contribution for high-risk patients.
*   $T_{target,i}$: The target time-to-treatment for high-risk call $i$.
*   $\text{FairnessPenalty}(E_{g,t}, \bar{E}_{g})$: A function penalizing deviation from equitable service. For example, it could be the sum of squared differences $(\frac{E_{g,t} - \bar{E}_{g}}{\bar{E}_{g}})^2$ for each demographic group $g$, where $E_{g,t}$ could be the average time-to-treatment or service rate for group $g$ over a rolling window.
*   $(1-C_i)$ is the uncertainty of call $i$. The term penalizes dispatching to unconfirmed calls, weighted by their uncertainty, discouraging over-committing resources to potentially false alarms.

**Constraints:**
1.  **Resource Allocation:** $\sum_{i} x_{i,h,t} \le 1 \quad \forall h, t$ (Each helicopter can only be dispatched to one call at a time).
2.  **One Dispatch Per Call:** $\sum_{h} x_{i,h,t} \le 1 \quad \forall i, t$ (Each call gets at most one helicopter).
3.  **Availability:** If helicopter $h$ is already en route or occupied, $x_{i,h,t}=0$.
4.  **No Dispatch if Not Confirmed (unless low uncertainty):** If $C_i < C_{threshold}$ (low confidence), then $\sum_h x_{i,h,t} \cdot y_{i,t} = 0$, meaning a helicopter can only be fully dispatched if it's confirmed. However, the last term in the objective allows for "speculative" dispatching to *unconfirmed* calls that have *some* likelihood of being real (i.e., $y_{i,t}=0$ but $x_{i,h,t}=1$), but heavily penalizes low-confidence dispatches.
5.  **Budget Constraint:** The total operational cost (fuel, maintenance, personnel) of dispatches must remain within the fixed budget. This is typically an overarching constraint for the entire operational period rather than a real-time decision variable.
6.  **Legal/Explainability:** Decisions must be justifiable based on the parameters above.

**Key Assumptions:**
*   The survival function $S_i(T)$ is monotonic decreasing with treatment time.
*   The weights $\lambda_1, \lambda_2, \lambda_3, \alpha$ accurately reflect the city's priorities.
*   We can reliably estimate $C_i$ (call genuineness).

**Trade-offs:**
*   **Lives Saved vs. Equity:** If a high-priority call comes from a well-served area, and a slightly lower-priority call comes from an underserved demographic group, the system might need to weigh the immediate survival benefit against the long-term goal of reducing disparities. $\lambda_1$ vs $\lambda_3$.
*   **Lives Saved vs. Robustness:** Dispatching immediately to a call with moderate uncertainty (potentially false) could save a life if real, but exhaust resources for later, genuine calls. The $\alpha$ term penalizes this, holding back resources, which could delay treatment for a real but suspicious call.
*   **Time-to-Treatment vs. Lives Saved:** While faster treatment generally means more lives saved, a strict time-to-treatment target might lead to inefficient dispatches (e.g., sending a helicopter further for a minor improvement in TTT, when it could be used for a higher-impact call elsewhere). $W_{risk,i}$ and $\lambda_2$ adjust this.

---

### B. Algorithmic Approach for Real-time Dispatch

**Approach: Rolling Horizon Optimization with Bayesian Inference and Adversarial Modeling**

**Core Algorithm:** A sequential decision-making process implemented as a **Rolling Horizon Optimization (RHO)**.
1.  **State Definition:** At each time step, the system's state includes:
    *   Current location and status (available, en route, at scene, returning) of all helicopters.
    *   List of active emergency calls (pending, dispatched, completed), their estimated severity, location, demographic group, and confidence score $C_i$.
    *   Predicted future resource availability (e.g., helicopter return times).
    *   Historical and real-time aggregate service metrics for each demographic group.
    *   Predicted traffic, weather, and other environmental factors affecting travel times.

2.  **Optimization Window:** The RHO considers a look-ahead window (e.g., 15-30 minutes) into the future. It attempts to optimize the objective function over this window, assuming future call arrivals follow a probabilistic distribution.

3.  **Decision Policy:** When a new call arrives or a helicopter becomes available:
    *   **Call Validation & Risk Assessment (Bayesian Inference):**
        *   **Initial Assessment:** Immediately classify the call based on metadata (caller ID patterns, call frequency, reported symptoms, location history).
        *   **Sensor Fusion:** Integrate real-time sensor data (e.g., traffic cameras, crash sensors, environmental monitors, IoT data from connected vehicles/infrastructure near the incident) to corroborate the call.
        *   **Confidence Score ($C_i$):** Use a Bayesian network or a deep learning model to output $C_i$, the probability that the call is genuine, given all available evidence. Update $C_i$ as new information arrives.
        *   **Risk Score ($R_p$):** Simultaneously, predict the patient's risk profile based on inferred medical urgency and contextual factors.

    *   **Adversarial Modeling & Detection:**
        *   **Adversary Model:** Assume adversaries aim to monopolize resources by placing false calls, potentially mimicking genuine patterns. Model adversary behavior using game theory (e.g., a Stackelberg game where the ADS is the leader, anticipating adversary moves) or statistical anomaly detection. Adversaries might adapt their strategies.
        *   **Detection:** Focus on identifying inconsistencies between reported events and sensor data, unusual call patterns (e.g., bursts from new numbers in specific areas, repeated calls for minor issues), and deviations from expected emergency behavior. A classifier trained on historical false calls, potentially augmented with adversarial examples, helps identify suspicious calls.
        *   **Mitigation (Integrated into Optimization):**
            *   **Verification Strategy:** For calls with $C_i < C_{threshold}$ (suspicious), the system prioritizes *verification* over immediate dispatch. This might involve dispatching a ground unit first, attempting a callback, or waiting for corroborating sensor data. The cost function penalizes dispatching helicopters to low-confidence calls, forcing the optimizer to consider the trade-off.
            *   **Resource Holding:** The optimizer might decide to hold back some helicopters if there's a high probability of future high-priority genuine calls or a significant number of suspicious active calls, to maintain robustness. This is implicitly handled by the RHO looking ahead and considering resource depletion.
            *   **Dynamic Re-routing:** If a dispatched helicopter is en route and the call is later confirmed false (e.g., ground unit reports no incident), the ADS immediately re-evaluates the fleet and re-routes the helicopter to the optimal base or a new high-priority real incident.

    *   **Dispatch Decision:**
        *   An integer linear program (ILP) or a heuristic search algorithm solves the objective function for the current RHO window. It considers:
            *   All pending calls (classified by urgency, risk, demographic group, confidence).
            *   Available helicopters, their current locations, and estimated travel times.
            *   The fairness penalty term, ensuring decisions don't exacerbate disparities.
            *   The penalty for dispatching to unconfirmed calls ($1-C_i$).
        *   The optimizer makes a dispatch decision for current calls and potentially schedules future re-positioning of idle helicopters.

**Complexity, Latency, and Deployment Considerations:**
*   **Complexity:** The ILP for optimizing helicopter allocation can be NP-hard. However, for a limited fleet (e.g., 5-10 helicopters) and a moderate number of simultaneous incidents (e.g., 10-20), efficient solvers can provide near-optimal solutions quickly. Heuristics (e.g., greedy dispatch followed by local search) can be used for larger scales or tighter latency constraints.
*   **Latency:** Critical decisions (dispatch/re-route) must occur within **1-5 seconds**. This requires:
    *   **Pre-computation:** Pre-calculate travel times, pre-train models.
    *   **Efficient Solvers:** Use highly optimized ILP solvers (e.g., Gurobi, CPLEX) or specialized heuristics.
    *   **Parallel Processing:** Run call classification, adversarial detection, and optimization in parallel.
*   **Deployment:**
    *   **Edge/Cloud Hybrid:** Sensor data processing and initial call classification could occur at the edge (local dispatch centers) for minimal latency. The central RHO and adversarial modeling would run on a robust, scalable cloud infrastructure.
    *   **Robust Communication:** High-bandwidth, low-latency, redundant communication channels between helicopters, ground units, dispatch centers, and the central ADS.
    *   **Fault Tolerance:** Implement hot-standby redundancy for critical components.
    *   **Human-in-the-Loop:** Dispatchers retain override capability. The system provides its rationale for decisions, fostering trust and enabling quick manual intervention if needed.

**Key Assumptions:**
*   Reliable, real-time sensor data is available.
*   Adversarial patterns are detectable, even if they evolve.
*   The number of simultaneous incidents and helicopters remains manageable for quick optimization.

**Trade-offs:**
*   **Responsiveness vs. Verification:** Immediate dispatch for every call is fastest but highly vulnerable to false alarms. Introducing verification steps improves robustness but adds latency. The $C_i$ score and $\alpha$ parameter directly manage this trade-off.
*   **Optimality vs. Speed:** A perfect global optimum might take too long. RHO with heuristics provides a good balance, yielding near-optimal solutions fast enough for real-time.
*   **Complexity vs. Explainability:** More complex models (e.g., deep learning for $C_i$) might be more accurate but harder to explain. Using interpretable models (e.g., Bayesian networks, rule-based systems) where possible, or employing explainable AI (XAI) techniques, helps maintain transparency.

---

### C. Data Pipeline and Privacy-Preserving Methods

**Data Sources & Pipeline:**

1.  **Real-time Sensor Feeds (Ingestion & Pre-processing):**
    *   **Source:** Traffic sensors (Waze, municipal loops), weather stations, helicopter telemetry (GPS, status, fuel), ground unit GPS, public safety IoT (e.g., gunshot detectors, crash sensors), aggregated environmental data (air quality monitors).
    *   **Processing:**
        *   **Ingestion:** Kafka/Pulsar for high-throughput stream processing.
        *   **Cleaning & Normalization:** Standardize units, handle missing values, remove outliers.
        *   **Geo-temporal Tagging:** Associate all data with precise location and timestamp.
        *   **Aggregation:** For privacy, aggregate relevant environmental/community data (e.g., average air quality per block, traffic density per road segment).
        *   **Anonymization:** If any sensor feed *could* implicitly identify individuals (e.g., a unique mobile device ID in a crowd sensor), anonymize/hash it immediately.
    *   **Output:** Real-time stream of clean, geo-tagged, contextual information.

2.  **Call Metadata (Ingestion & Feature Extraction):**
    *   **Source:** 911 dispatch system, emergency call center (CAD).
    *   **Processing:**
        *   **Ingestion:** API integration or secure database access.
        *   **PII Removal/Pseudonymization:** **Crucially, raw personal medical records are off-limits.**
            *   Caller ID: Hashed to track repeat callers without revealing identity.
            *   Reported symptoms: NLP techniques extract keywords (e.g., "chest pain," "difficulty breathing," "car accident") and categorize incident types/severities. No attempt to infer personal medical history beyond immediate reported symptoms.
            *   Reported Age/Gender: If provided, immediately map to broad categories (e.g., "child," "adult," "elderly") and *do not store exact values*.
            *   Location: Geocode incident address.
        *   **Feature Engineering:** Call frequency from location/caller ID, time of day, day of week, keywords extracted.
    *   **Output:** Structured features for each call, linked to incident ID (not PII).

3.  **Historical Anonymized Aggregated Outcomes (Batch Processing & Model Training):**
    *   **Source:** Hospital records, pre-hospital care reports, city mortality data.
    *   **Processing:**
        *   **Anonymization:** All direct identifiers (names, exact birthdates, addresses) are removed. Indirect identifiers are suppressed, generalized, or perturbed (e.g., k-anonymity, differential privacy for counts).
        *   **Aggregation:** Data is aggregated by incident type, severity level, rough location (e.g., census tract, neighborhood), broad demographic groups (e.g., age bands, income quartiles from census data *not* individual PII), and treatment outcomes (survival, recovery time).
        *   **Derived Features:** Survival rates per incident type/demographic, average time-to-treatment by region, resource utilization patterns.
    *   **Output:** Clean, aggregated datasets for model training and fairness baseline establishment.

**Privacy-Preserving Methods:**

1.  **Data Minimization:** Only collect and process data strictly necessary for the ADS's objectives. Avoid any data that could infer personal medical history or identity beyond the immediate emergency context.
2.  **Strict Anonymization & Pseudonymization:**
    *   **At Source:** Implement anonymization as early as possible in the pipeline.
    *   **Pseudonymization:** For internal linking of call data to outcomes (e.g., for model training), use pseudonyms (e.g., cryptographic hashes of incident IDs) instead of direct identifiers.
    *   **K-anonymity/L-diversity:** For aggregated datasets, ensure that each record is indistinguishable from at least k other records, or that sensitive attributes have at least L distinct values within each group, preventing re-identification.
3.  **Differential Privacy (for Public Reports/Model Release):**
    *   When releasing *any* aggregate statistics derived from sensitive data (e.g., average time-to-treatment per specific demographic group if the group size is small), inject controlled noise to provide a provable guarantee against re-identification, without unduly compromising utility. This is more relevant for external reporting than for internal real-time operations.
4.  **Secure Multi-Party Computation (SMC) / Federated Learning (Advanced, if absolutely necessary):**
    *   If there's a need to train a model using sensitive data residing in different silos (e.g., hospital A and hospital B) without sharing the raw data, SMC or Federated Learning could be considered. This would allow joint model training on encrypted or local datasets. This is usually too complex and latent for real-time dispatch but could be used for the *offline* training of some models. For this specific scenario, given "raw personal medical records are off-limits," strong aggregation and anonymization should suffice.
5.  **Access Control and Encryption:**
    *   **Role-Based Access Control (RBAC):** Strict permissions based on roles, ensuring only authorized personnel can access different data layers.
    *   **Encryption at Rest and In Transit:** All data (raw, processed, and models) must be encrypted using industry-standard protocols.
6.  **Data Governance and Auditing:**
    *   **Clear Policies:** Document data retention, usage, and sharing policies.
    *   **Regular Audits:** Conduct periodic security and privacy audits to ensure compliance and identify vulnerabilities.
    *   **Data Protection Officer (DPO):** Appoint a DPO to oversee privacy practices.

**Minimal Aggregated or Derived Features Necessary:**

1.  **Geospatial Risk Maps (Aggregated):**
    *   Heatmaps of historical emergency incident density, severity, and mortality rates per grid cell or census tract.
    *   Maps of vulnerable population density (e.g., high elderly population, low-income areas) derived from anonymized census data.
    *   Real-time traffic congestion, road closures, and weather impact maps.
2.  **Demographic Service Baselines (Aggregated & Anonymized):**
    *   Historical average time-to-treatment, dispatch rates, and (adjusted) mortality outcomes per defined demographic group (e.g., by zip code, income quartile, age band). Used for the fairness penalty term $\text{FairnessPenalty}$.
3.  **Real-time Resource Status:**
    *   Number of available helicopters, their locations, and estimated return-to-base times.
    *   Status of ground units (e.g., for call verification).
4.  **Incident Likelihood & Severity Scores:**
    *   $C_i$: Probability of a call being genuine (vs. adversarial false call).
    *   $R_p$: Patient risk score/severity.
    *   Predicted survival probability $S_i(T)$ as a function of incident type and treatment time.
5.  **Adversarial Threat Indicators:**
    *   Real-time detection patterns of suspicious call frequencies, origins, and content; historical false call patterns.

**Key Assumptions:**
*   City officials and legal frameworks support the use of anonymized and aggregated data for public safety.
*   Reliable mechanisms for securely integrating data from various public safety agencies and sensors exist.
*   Adequate technical resources (encryption, secure storage, specialized personnel) are available.

**Trade-offs:**
*   **Data Utility vs. Privacy:** More granular data might improve prediction accuracy, but higher granularity increases re-identification risk. The chosen approach prioritizes privacy through strong anonymization and aggregation, accepting a potential marginal reduction in model precision for extreme edge cases.
*   **Computational Cost vs. Privacy Guarantee:** Techniques like SMC offer strong privacy but are computationally intensive and might not be suitable for real-time processing. Differential privacy is more practical for aggregate reporting.

---

### D. Evaluation, Validation, and Continuous Monitoring Plan

**Pre-Deployment Evaluation & Validation:**

1.  **Simulation & Retrospective Analysis:**
    *   **Method:** Use years of historical anonymized call data, sensor feeds, and outcome data to simulate the ADS's performance offline. Compare ADS decisions (dispatch choices, treatment times, predicted outcomes) against actual historical outcomes and the current baseline.
    *   **Metrics:**
        *   **Effectiveness:**
            *   **Expected Lives Saved (ELS):** Calculate $\sum_{i} S_i(T_{ADS\_dispatch}) - \sum_{i} S_i(T_{baseline\_dispatch})$. This is a counterfactual estimate.
            *   Average time-to-treatment (TTT) and TTT for high-risk patients.
            *   Resource utilization (e.g., helicopter active time, empty repositioning).
        *   **Fairness:**
            *   Disparity in average TTT across demographic groups.
            *   Disparity in proportion of calls dispatched by severity/group.
            *   Disparity in ELS across demographic groups (adjusting for baseline differences in health outcomes).
            *   Metrics like Gini coefficient or Theil index on TTT or ELS by group.
        *   **Safety/Robustness:**
            *   False Positive Rate (FPR) of adversarial call detection (i.e., genuine call flagged as false).
            *   False Negative Rate (FNR) of adversarial call detection (i.e., false call flagged as genuine).
            *   Number of incidents where resources were exhausted due to false calls.
    *   **Statistical Tests:** Compare distributions of TTT, ELS, etc., using t-tests or ANOVA (if data is normal) or non-parametric tests (Mann-Whitney U, Kruskal-Wallis) for different scenarios (ADS vs. baseline, or across demographic groups).

2.  **Human-in-the-Loop Validation (Shadow Mode):**
    *   **Method:** Run the ADS in "shadow mode" parallel to human dispatchers for 3-6 months. The ADS makes recommendations, but human dispatchers make the actual decisions. This allows observation of discrepancies and collection of feedback without operational risk.
    *   **Metrics:**
        *   **Agreement Rate:** Percentage of ADS recommendations accepted by human dispatchers.
        *   **Override Rate:** Percentage of ADS recommendations overridden by humans, with reasons for override.
        *   **Latency Impact:** Measure if ADS processing adds significant delay to overall decision time.
    *   **Qualitative Feedback:** Conduct interviews and surveys with dispatchers to understand usability, explainability, and perceived trust.

**Post-Deployment Continuous Monitoring:**

All pre-deployment metrics will be continuously monitored.

1.  **Effectiveness Monitoring:**
    *   **Primary Metric:** **Observed Lives Saved (OLS) / Mortality Reduction.** Compare city-wide mortality rates for relevant emergency types post-ADS deployment against the pre-ADS baseline, using **causal inference techniques** (e.g., Difference-in-Differences, Synthetic Control) to account for confounding factors and seasonality.
        *   **Threshold:** A statistically significant reduction in mortality (e.g., p < 0.05, or a specific effect size like a 5% reduction). If no reduction is observed after 6-12 months, trigger a system review.
    *   **Secondary Metrics:**
        *   **Average TTT and TTT for High-Risk Patients:** Monitor moving averages and percentiles.
            *   **Threshold:** TTT for high-risk patients should consistently be below a predefined target (e.g., < 10 minutes average response), and the 90th percentile should show improvement over baseline.
        *   **Resource Utilization Efficiency:** Track helicopter idle time, mission success rate, and re-routing efficiency.
            *   **Threshold:** >X% active time, <Y% empty repositioning.
    *   **Statistical Tests:** Control charts (e.g., EWMA for mean TTT) to detect shifts. Regression models for OLS to control for covariates.

2.  **Fairness Monitoring:**
    *   **Metrics:**
        *   **Disparity in TTT:** Calculate the absolute or relative difference in mean TTT between demographic groups.
        *   **Disparity in Dispatch Rate:** Ratio of dispatch rates for similar incidents across groups.
        *   **Disparity in (Adjusted) Mortality/Survival:** Compare group-specific ELS/OLS, adjusting for baseline health disparities.
        *   **Concentration Indices:** (e.g., Gini) on service levels.
    *   **Statistical Tests:**
        *   **Two-sample t-tests/ANOVA:** To compare means of TTT, dispatch rates, or adjusted survival between groups.
        *   **Chi-squared tests:** For categorical disparities (e.g., dispatch vs. no dispatch across groups).
        *   **Regression Analysis:** To check for significant demographic coefficients in outcome models after controlling for other factors.
    *   **Thresholds:**
        *   No more than X% (e.g., 5-10%) difference in average TTT between any two demographic groups, relative to city-wide average.
        *   Adjusted mortality outcomes should not show a statistically significant worsening for any group compared to baseline.
        *   Regular alerts if any group falls below a predefined service equity threshold for a sustained period (e.g., 1 month).

3.  **Safety & Robustness Monitoring:**
    *   **Metrics:**
        *   **Adversarial Call Detection Performance:** Monitor FPR (false flagging genuine calls) and FNR (missing false calls) in real-time, using human verification logs as ground truth.
        *   **System Uptime & Latency:** Monitor system availability and decision-making latency.
        *   **Resource Exhaustion Events:** Track instances where all helicopters were unavailable when a high-priority call arrived.
        *   **Unconfirmed Dispatch Rate:** Rate of helicopter dispatches to calls with $C_i < C_{threshold}$.
    *   **Statistical Tests:** Control charts for FPR/FNR. Outlier detection for latency spikes.
    *   **Thresholds:**
        *   Adversarial FNR < 1% (prioritize catching false calls). FPR < 0.1% (prioritize not missing real calls).
        *   System uptime > 99.9%. Decision latency < 5 seconds 99% of the time.
        *   Resource exhaustion events < 1 per month.
        *   Unconfirmed dispatch rate should remain low (e.g., <5%).

**Key Assumptions:**
*   Reliable ground truth data for adversarial calls (via human follow-up) exists for ongoing model performance evaluation.
*   The city has a robust data collection and warehousing infrastructure for monitoring.
*   The baseline mortality rates and demographic data are accurate.

**Trade-offs:**
*   **Granularity of Monitoring vs. Cost:** Monitoring every metric at high frequency can be expensive. Prioritize key metrics for continuous, real-time tracking, while others are reviewed periodically.
*   **Statistical Power vs. Timeliness:** Detecting subtle shifts in fairness or effectiveness requires sufficient data, which might take time. Balance early warning systems (potentially with higher false alarm rates) with robust statistical confidence.

---

### E. Anticipated Failure Modes and Mitigation Strategies

**1. Technical Failures**

*   **Failure Mode:** **Sensor Failure/Degradation.** e.g., traffic data feed goes down, helicopter GPS malfunction, weather station offline.
    *   **Detection:**
        *   **Anomaly Detection:** Real-time monitoring of data streams for sudden drops, zero values, values outside plausible ranges, or lack of updates.
        *   **Cross-Verification:** Compare data from redundant sensors or alternative sources (e.g., comparing traffic camera data with Waze).
        *   **Heartbeat Checks:** Regular pings to data sources.
    *   **Mitigation:**
        *   **Redundancy & Failover:** Use multiple data sources for critical information. If primary fails, switch to secondary.
        *   **Imputation/Fallback:** For missing data, fall back to historical averages (e.g., average traffic for that time/day) or use predictive models based on available correlated data.
        *   **Graceful Degradation:** The ADS operates with reduced accuracy rather than shutting down. If a helicopter's GPS fails, it might be temporarily excluded from optimal dispatch until fixed, or assigned to lower-priority missions with manual oversight.
        *   **Alerting:** Automated alerts to technicians and human dispatchers for manual intervention.

*   **Failure Mode:** **Model Drift/Degradation.** The underlying patterns change (e.g., new incident types, changing demographics, adversary tactics evolve), and the survival prediction model, adversarial detection, or risk assessment model becomes less accurate.
    *   **Detection:**
        *   **Performance Monitoring:** Continuously monitor model prediction accuracy against human-labeled ground truth or verified outcomes (see D). Track metrics like precision, recall, F1-score, calibration.
        *   **Data Distribution Monitoring:** Monitor input data distributions for shifts (e.g., new call symptom keywords, changes in false call characteristics).
    *   **Mitigation:**
        *   **Continuous Retraining:** Implement automated (or semi-automated with human review) pipelines for retraining models with fresh, labeled data at regular intervals (e.g., weekly/monthly).
        *   **A/B Testing New Models:** Before deploying a retrained model, A/B test it in shadow mode against the active model to ensure improvement.
        *   **Adaptation to Adversaries:** Incorporate new adversarial patterns into training data, possibly using Generative Adversarial Networks (GANs) to generate synthetic adversarial examples.
        *   **Human Override:** Empower dispatchers to override potentially erroneous ADS decisions, providing feedback for model improvement.

*   **Failure Mode:** **System Outage/Downtime.** The ADS application or its infrastructure crashes.
    *   **Detection:**
        *   **Health Checks:** Automated monitoring of all system components (databases, servers, APIs, network) with heartbeat signals.
        *   **Alerting:** Immediate alerts to system administrators and dispatchers.
    *   **Mitigation:**
        *   **High Availability:** Redundant servers (active-active or active-passive failover) across different geographical regions.
        *   **Disaster Recovery Plan:** Detailed procedures for restoring services from backups.
        *   **Automated Rollback:** Mechanism to revert to a previous stable version of the software if a new deployment causes issues.
        *   **Manual Fallback:** Clearly defined human emergency dispatch protocols that can be immediately activated in case of full system failure.

**2. Behavioral Failures (User & Adversary)**

*   **Failure Mode:** **Increased Adversarial Sophistication.** Adversaries learn how the ADS works and adapt their false call patterns to evade detection.
    *   **Detection:**
        *   **Feedback Loops:** Human dispatchers flag previously undetected false calls, providing ground truth for retraining.
        *   **Pattern Analysis:** Advanced anomaly detection on call data (e.g., using unsupervised learning) to identify novel attack patterns.
        *   **Honeypots/Simulations:** Deploy simulated ADS environments to probe for new attack vectors.
    *   **Mitigation:**
        *   **Adaptive Adversarial Models:** Continuously update the adversarial detection models with new data and use machine learning techniques robust to concept drift.
        *   **Multi-Factor Verification:** Increase the number of independent verification sources required for dispatch to suspicious calls.
        *   **Deterrence:** Collaborate with law enforcement to track and prosecute perpetrators of false calls.
        *   **"Cognitive Overload" Strategy:** If adversaries try to overwhelm the system, have policies to de-prioritize high-frequency, low-evidence calls from specific patterns.

*   **Failure Mode:** **Operator Over-reliance (Automation Bias).** Human dispatchers blindly trust ADS recommendations, potentially missing errors or losing critical decision-making skills.
    *   **Detection:**
        *   **Override Monitoring:** Track the frequency and reasons for human overrides. A decreasing override rate or a lack of overrides even in clearly ambiguous situations might signal over-reliance.
        *   **Performance Drills:** Periodic simulated emergency scenarios where dispatchers are tested, sometimes with intentionally flawed ADS recommendations.
    *   **Mitigation:**
        *   **Meaningful Human Control:** Design the UI to clearly present the ADS's rationale, highlighting uncertainties or trade-offs.
        *   **Mandatory Review Points:** For high-stakes decisions or low-confidence ADS recommendations, require explicit human confirmation.
        *   **Training & Education:** Regular training on ADS capabilities, limitations, and how to effectively use and challenge its recommendations. Emphasize critical thinking.
        *   **Feedback Mechanisms:** Make it easy for dispatchers to provide feedback on ADS decisions, including flagging incorrect ones.

**3. Social Failures (Unintended Societal Impact)**

*   **Failure Mode:** **Exacerbated Disparities / Algorithmic Bias.** The ADS unintentionally prioritizes certain demographic groups over others or perpetuates historical biases present in the training data, worsening existing inequities.
    *   **Detection:**
        *   **Continuous Fairness Monitoring:** As detailed in section D, regularly compare fairness metrics (TTT, dispatch rates, adjusted outcomes) across all defined demographic groups.
        *   **External Audits:** Engage independent fairness auditors to review the system and its impact.
        *   **Community Feedback:** Establish channels for community members to report perceived inequities or issues.
    *   **Mitigation:**
        *   **Fairness-Aware Optimization:** Adjust the $\lambda_3$ and $\text{FairnessPenalty}$ terms in the objective function to more aggressively penalize disparities.
        *   **Bias Mitigation Techniques:** Apply post-processing (e.g., re-calibration, equalizing odds) or in-processing (e.g., adversarial debiasing) techniques to reduce bias in prediction models.
        *   **Data Re-weighting/Re-sampling:** During model training, oversample underrepresented groups or re-weight training examples to reduce bias.
        *   **"Equity Audits":** Periodically review the data, models, and system decisions specifically for disparate impacts.
        *   **Transparency:** Be open about the data used, the models' limitations, and the measures taken to promote fairness.

*   **Failure Mode:** **Public Distrust/Misunderstanding.** Lack of transparency, perceived unfairness, or unexplained decisions erode public confidence in the system.
    *   **Detection:**
        *   **Media Monitoring:** Track public sentiment and news coverage.
        *   **Public Feedback Channels:** Monitor complaints, inquiries, and public forums.
        *   **Surveys:** Periodically survey public perception of emergency services and the ADS.
    *   **Mitigation:**
        *   **Transparency and Explainability:** Publish high-level, accessible information about how the ADS works, its objectives, and the safeguards in place. Provide simplified explanations for typical decisions (while protecting privacy and security).
        *   **Community Engagement:** Involve community leaders and representatives in the ADS design, oversight, and feedback processes.
        *   **Public Education Campaigns:** Clearly communicate the benefits, limitations, and operational details of the system.
        *   **Clear Appeal Process:** Establish a transparent process for citizens to appeal or inquire about specific service outcomes if they believe the ADS made an incorrect or unfair decision (maintaining privacy).

**Key Assumptions:**
*   Adequate resources are allocated for ongoing monitoring, retraining, and system improvements.
*   Human operators are willing and able to participate in feedback loops and continuous training.
*   A legal and ethical framework is in place to support these mitigation strategies.

**Trade-offs:**
*   **Robustness vs. Cost:** Implementing all redundancies and advanced detection mechanisms can be expensive. Prioritize based on criticality.
*   **Safety vs. False Alarms:** Setting detection thresholds too strictly can lead to missed genuine calls (high FNR) or too many false alarms, desensitizing operators.
*   **Efficiency vs. Explainability:** Highly optimized, black-box models may be efficient but hard to explain. Balance between model performance and the ability to justify decisions.

---

### F. Minimal Clarifying Questions for City Officials

These questions aim to resolve underspecification, set concrete boundaries, and ensure alignment with city priorities and legal frameworks.

1.  **Legal & Ethical Framework for Data and Demographics:**
    *   "What are the precise legal definitions and reporting requirements for 'demographic groups' in this city (e.g., race, ethnicity, income bracket, age bands, geographic regions like census tracts, disability status)? How can we legally access *anonymized and aggregated* historical data for these groups, and what are the strict privacy and data governance rules we must adhere to, especially regarding inferred patient information?"
        *   *Justification:* Crucial for defining the fairness objective ($\lambda_3$) and ensuring legal compliance for the entire data pipeline (Part C). Without this, "equitable service" is ill-defined.

2.  **Definition of "High-Risk Patient" and Survival Model Data:**
    *   "How do you currently define a 'high-risk patient' for helicopter dispatch, and what specific clinical indicators, contextual factors, or existing scores (e.g., Glasgow Coma Scale for trauma) should the system prioritize? Can we access historical anonymized outcome data linked to these indicators and incident types to build and validate our survival probability model $S_i(T)$?"
        *   *Justification:* Essential for operationalizing objectives (1) and (2) and building the core predictive models.

3.  **Adversarial Threat Profile & Response Protocols:**
    *   "What are the specific patterns, frequencies, and motivations of adversarial actors placing false calls that you've observed historically? What are the existing protocols for verifying suspicious calls, and what is the maximum acceptable delay introduced by verification steps before dispatching a helicopter for a potentially genuine, but suspicious, call?"
        *   *Justification:* Directly informs the adversarial modeling (Part B), the confidence score $C_i$, and the robustness trade-offs. Helps set realistic thresholds for $\alpha$ and verification procedures.

4.  **Operational Latency and Human-in-the-Loop Mandates:**
    *   "What is the maximum acceptable end-to-end latency for a dispatch decision from the moment a call is received, including any mandatory human oversight or verification steps? Are there specific types of decisions where human override or explicit approval is legally or operationally mandated, and what should be the process for these overrides?"
        *   *Justification:* Sets critical performance requirements for the algorithmic approach (Part B) and shapes the deployment strategy, including the human-in-the-loop design.

5.  **Fixed Budget Allocation and Resource Constraints:**
    *   "What is the precise fixed budget allocated for the *entire lifecycle* of this ADS (development, deployment, continuous operation, maintenance, and personnel for oversight/data labeling)? What is the exact number of helicopters available, their operational constraints (e.g., flight hours, maintenance schedules), and are there any ground support units whose dispatch we should also consider optimizing?"
        *   *Justification:* Fundamental constraint for feasibility and scope of the entire project, impacting all parts (A-E). It defines the available "fleet" for the decision variables.

6.  **Interoperability and Existing Infrastructure:**
    *   "How will this ADS integrate with existing emergency services (e.g., 911 dispatch, ground ambulance systems, police)? What are the existing communication protocols, data formats, and data sharing agreements in place with these agencies, and what level of real-time data exchange can we expect?"
        *   *Justification:* Crucial for designing the data pipeline (Part C) and ensuring seamless operation within the city's broader emergency response ecosystem. This impacts the quality and availability of real-time sensor and call metadata.

In [10]:
deepseek = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com/v1")
model_name = "deepseek-chat"

response = deepseek.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

# **Automated Decision System for Emergency Medical Helicopter Dispatch**

## **A. Formal Objective & Decision Variables**

**Decision Variables:**
- \( x_{ijt} \in \{0,1\} \): Dispatch helicopter \(i\) to incident \(j\) at time \(t\)
- \( y_{jt} \in \{0,1\} \): Classify incident \(j\) at time \(t\) as requiring helicopter
- \( r_{gt} \in \mathbb{R}^+ \): Reserve capacity allocated to demographic group \(g\) at time \(t\)

**Multi-Objective Formulation (Lexicographic with ε-constraints):**

1. **Primary Objective (Lives Saved):**
   \[
   \max \sum_{j,t} \mathbb{E}[\Delta M_{jt}] \cdot y_{jt}
   \]
   where \(\Delta M_{jt} = \text{mortality reduction estimate for incident } j\)

2. **Secondary Objectives (as ε-constraints):**
   - **Time-to-treatment:** \(\mathbb{P}(\text{TTT}_{high-risk} > T_{max}) \leq \alpha\)
   - **Equity:** \(\frac{\text{Service Rate}_g}{\text{Need}_g} \geq 1 - \delta \ \forall g\)
   - **Robustness:** \(\mathbb{E}[\text{Loss}|\text{false calls}] \leq \beta\)
   - **Budget:** \(\sum \text{Cost}(x_{ijt}) \leq B\)

**Key Assumptions & Trade-offs:**
1. Mortality reduction can be estimated from noisy data with known confidence intervals
2. Equity is defined as *proportionality* (service rate ∝ medical need), not equality
3. 5% degradation in primary objective acceptable for substantial fairness/robustness gains
4. Explainability requires linear or monotonic relationships in final scoring

## **B. Algorithmic Approach**

**Core Algorithm: Two-Stage Robust Optimization with Real-Time Learning**

**Stage 1: Incident Risk Scoring (performed every 30 seconds)**
```
Risk_Score(j) = w₁·f₁(sensor_evidence) + w₂·f₂(call_metadata) 
                + w₃·f₃(historical_pattern) - w₄·g(adversarial_indicators)
```
where \(f_i\) are calibrated probabilistic models.

**Stage 2: Dispatch Optimization (Mixed-Integer Program with uncertainty sets):**
\[
\min_{x,y} \sum_j (1 - p_j^{genuine}) \cdot c_j \cdot y_{jt} + \lambda \cdot \text{Unfairness}(x)
\]
subject to:
- Helicopter capacity constraints
- Uncertainty set for travel times (weather, traffic)
- Adversarial response: \(p_j^{genuine} = h(\text{location}, \text{caller_history}, \text{recent_patterns})\)

**Adversary Modeling:**
- **Behavioral model:** Assume rational adversary minimizing genuine calls served
- **Learning:** Online contextual bandit with Thompson sampling to detect exploitation patterns
- **Defense:** Incorporate cost for verifying suspicious calls via ground units

**Complexity & Latency:**
- Greedy heuristic with optimality guarantee within 15% for real-time (<60s decision)
- Full optimization re-run every 5 minutes
- Computational complexity: \(O(nm\log m)\) where \(n\)=incidents, \(m\)=helicopters

**Deployment:**
- Gradual rollout with shadow mode for 2 months
- Human-in-the-loop override with mandatory justification logging
- Fallback to current protocol if confidence <80%

## **C. Data Pipeline & Privacy Methods**

**Minimal Feature Set:**
1. **Anonymized Call Metadata:** Grid location (100m²), time, caller device type, call duration
2. **Sensor Feeds (no PII):** Acoustic analysis (background sounds), traffic camera aggregates (vehicle count, speed), weather station data
3. **Historical Aggregates:** Mortality rates by grid×time×incident_type (k-anonymized with k=25)
4. **Derived Features:** Time to nearest hospital (ground/air), recent call frequency in area

**Privacy-Preserving Techniques:**
- **Differential privacy** (ε=0.5) on all aggregated statistics
- **Federated learning** for model updates across hospitals without data sharing
- **Homomorphic encryption** for sensitive computations on encrypted call metadata
- **Data minimization:** Raw audio discarded after 24h, only features retained

**Pipeline Architecture:**
```
Raw Streams → [DP Noise Injection] → Feature Extraction → Model Serving
             ↓
Aggregated Statistics → [k-Anonymization] → Historical Database
```

## **D. Evaluation & Monitoring Plan**

**Pre-deployment Validation:**
1. **Effectiveness:** Simulated mortality reduction vs current baseline (target: ≥15% improvement)
2. **Fairness:** Disparate impact ratio test: \(\min_g(\frac{Service_g}{Need_g}) \geq 0.8\)
3. **Robustness:** Stress tests with 30% false call rate (target: <10% performance degradation)

**Continuous Monitoring (Real-time Dashboard):**

| Metric | Threshold | Action |
|--------|-----------|--------|
| Mortality gap (simulated vs actual) | >5% | Model recalibration |
| Demographic disparity (KL divergence) | >0.1 | Review allocation rules |
| False positive rate (helicopter dispatch) | >20% | Adversary model update |
| 95th percentile response time | >25min | Capacity review |

**Statistical Tests:**
- Weekly: **McNemar's test** for performance drift
- Monthly: **Bayesian A/B testing** for policy changes
- Quarterly: **Counterfactual fairness audit** using synthetic data

## **E. Anticipated Failure Modes & Mitigations**

**Technical Failures:**
1. **Sensor spoofing:** Adversaries mimic emergency sounds
   - *Detection:* Anomaly detection on sensor correlation
   - *Mitigation:* Require multi-sensor confirmation for high-risk areas

2. **Model collapse:** Distribution shift in call patterns
   - *Detection:* KL divergence >0.2 between training/operational data
   - *Mitigation:* Online learning with conservative updates (η=0.01)

**Behavioral/Social Failures:**
1. **Equity paradox:** Over-correction creates new disparities
   - *Detection:* Significant (p<0.01) reversal in previously underserved groups
   - *Mitigation:* Smooth allocation changes, community feedback integration

2. **Gaming escalation:** Adversaries adapt to detection
   - *Detection:* Increasing false negative rate for genuine calls
   - *Mitigation:* Randomized reserve allocation, periodic strategy refresh

**Rollback Strategy:**
- Tiered fallback: (1) Conservative mode (2) Human mediation (3) Revert to legacy system
- Decision trigger: Two consecutive weeks of performance below baseline

## **F. Clarifying Questions for City Officials**

1. **Legal/Explainability Constraints:**
   - "What specific regulations govern automated dispatch decisions?"
   - "What level of explanation is required for individual decisions (feature importance vs full model logic)?"

2. **Equity Definition:**
   - "Is the priority equal outcomes, equal access, or proportional-to-need?"
   - "Which protected attributes can we use for monitoring (legally and ethically)?"

3. **Resource Constraints:**
   - "What is the maximum acceptable increase in operational costs?"
   - "Can we temporarily reallocate ground ambulances as a buffer system?"

4. **Validation Baseline:**
   - "What is the current performance baseline (mortality rates, response times by demographic)?"
   - "What historical data is available for control group comparisons?"

5. **Adversarial Context:**
   - "What is the historical frequency and pattern of false calls?"
   - "Are there known groups or areas with systematic exploitation attempts?"

---

**Key Trade-offs Acknowledged:**
1. **Lives vs Fairness:** Accept up to 5% fewer total lives saved for ≥20% reduction in disparity
2. **Accuracy vs Explainability:** Use interpretable models (gradient boosting) over deep learning despite potential 2-3% accuracy loss
3. **Responsiveness vs Robustness:** 30-second verification delay for suspicious calls to reduce false dispatches by 40%
4. **Privacy vs Utility:** 15% noisier mortality estimates due to differential privacy requirements

This design prioritizes **provable robustness** and **auditable fairness** while maintaining life-saving efficacy through adaptive learning and explicit trade-off management.

In [11]:
# Updated with the latest Open Source model from OpenAI

groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
model_name = "openai/gpt-oss-120b"

response = groq.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)


Below is a **complete design blueprint** for the Emergency‑Medical‑Helicopter Automated Decision‑Making System (EMS‑ADS).  
The presentation follows the exact six headings you asked for and, for each heading, lists **assumptions**, the **core design choices**, the **trade‑offs being made**, and **justifications** that tie back to the priority list (lives‑saved → time‑to‑treatment → equity → robustness → budget → explainability).

---

## (A)  Formal Objective & Decision Variables  

### 1. Decision variables  

| Symbol | Meaning | Domain |
|--------|---------|--------|
| \(x_{i,t} \in \{0,1\}\) | “Dispatch helicopter *h* to incident *i* at decision epoch *t*” (1 = dispatch, 0 = not) | binary |
| \(h\in\mathcal{H}\) | Set of helicopters (| \(\mathcal{H}\) | = fleet size) | – |
| \(i\in\mathcal{I}_t\) | Set of incident calls received in the rolling horizon at epoch *t* | – |
| \(d_i\in\mathcal{D}\) | Demographic group of the patient (e.g. age‑bucket × race × income‑zone) – **coarse, non‑identifying** | categorical |
| \(s_i\in[0,1]\) | Estimated clinical severity (probability of death without helicopter) – derived from sensor feed & call metadata | continuous |
| \(p_i\in[0,1]\) | Posterior probability that incident *i* is a false (adversarial) call | continuous |
| \(y_i\in\{0,1\}\) | Ground‑truth “true emergency” (known only retrospectively for training/validation) | binary |

### 2. Core building blocks  

* **Expected lives saved (ELS)** for a dispatch decision:  
  \[
  \text{ELS}_i = x_{i,t}\; \bigl( r(s_i)\, (1-p_i) \bigr)
  \]
  where \(r(s_i)\) is a calibrated mortality‑reduction function (e.g. logistic: \(r(s)=\alpha \frac{s}{s+\beta}\)). It maps severity to the *expected* reduction in the probability of death if the helicopter arrives within the clinical “golden‑hour”.

* **Time‑to‑treatment penalty (TTP)** for high‑risk patients (severity above a policy threshold \(s^{\star}\)):  
  \[
  \text{TTP}_i = x_{i,t}\; \bigl( \max\{0, \tau_{i}^{\text{arrival}}- \tau^{\text{target}} \}\bigr)\; \mathbf{1}_{\{s_i\ge s^{\star}\}}
  \]
  where \(\tau_{i}^{\text{arrival}}\) is the predicted arrival time (including travel, loading, weather) and \(\tau^{\text{target}}\) is the clinically‑defined maximum (e.g. 15 min).

* **Equity penalty (EP)** – we adopt **disparity‑aware weighted fairness**: each demographic group \(d\) has a target service level \(\bar{T}_d\) (e.g., median response time). Deviation from the target is penalised:  
  \[
  \text{EP}_d = \bigl|\;\overline{\tau}_{d} - \bar{T}_d\;\bigr|
  \]
  where \(\overline{\tau}_{d}\) is the observed average response time for group \(d\) over the past evaluation window.

* **Robustness penalty (RP)** – captures risk of allocating to a false call:  
  \[
  \text{RP}_i = x_{i,t}\; \lambda_{\text{adv}}\,p_i
  \]
  with \(\lambda_{\text{adv}}>0\) a tunable coefficient (higher → more conservative).

* **Budget / capacity constraints** (fixed fleet, crew duty‑time, fuel):  

  \[
  \sum_{i\in\mathcal{I}_t} x_{i,t}\; \mathbf{1}_{\{h\text{ assigned to }i\}}\;\le\; 1\quad\forall h\in\mathcal{H}
  \]  

  \[
  \sum_{i\in\mathcal{I}_t} x_{i,t}\; \tau^{\text{flight}}_{i}\;\le\; B_{\text{crew}} \quad\text{(crew‑hour budget per shift)}
  \]

### 3. Multi‑objective formulation  

We embed the four priority levels into a **single scalarised objective** using a lexicographic weighting that can be tuned but preserves the ranking of the priorities:

\[
\begin{aligned}
\max_{x} \; & \underbrace{\sum_{i} \text{ELS}_i}_{\text{(1) Lives saved}} \\
& -\;\underbrace{w_{T}\sum_{i}\text{TTP}_i}_{\text{(2) Time‑to‑treatment}}\\
& -\;\underbrace{w_{E}\sum_{d}\text{EP}_d}_{\text{(3) Equity}}\\
& -\;\underbrace{w_{A}\sum_{i}\text{RP}_i}_{\text{(4) Adversarial robustness}}\\
\text{s.t. } & \text{capacity \& budget constraints} \\
& \text{explainability rule: } x_{i,t}=0\; \text{if } \text{model confidence}<\theta_{\text{exp}} .
\end{aligned}
\]

*Weights* \(w_T, w_E, w_A\) are set so that any feasible increase in lives saved dominates any possible loss in the lower‑priority terms (lexicographic ordering). In practice we implement this with **ε‑constraint**: first maximise lives saved; then, subject to a small tolerance (e.g., ≤ 1 % loss), minimise the weighted sum of the lower‑priority penalties.

---

## (B)  Real‑Time Dispatch Algorithm  

### 1. High‑level architecture  

1. **Pre‑processing** (≤ 50 ms): ingest sensor & call metadata, compute severity score \(s_i\) and false‑call probability \(p_i\) via a *Bayesian classifier* (see adversary model).  
2. **Scenario generation** (≤ 100 ms): create a *finite set* \(\mathcal{S}\) of plausible world states (different realizations of \(p_i\) and travel‑time forecasts).  
3. **Robust stochastic optimisation** (≈ 300 ms): solve the mixed‑integer linear program (MILP) derived from the scalarised objective for each scenario, then pick the decision that maximises the **worst‑case** (or CVaR) objective across scenarios.  
4. **Dispatch & logging** (≤ 50 ms): send the chosen assignment to the helicopter crew and store the full decision trace for audit.

Overall latency ≈ 500 ms – well under the typical 2 s decision window required for EMS.

### 2. Modelling uncertainty & adversaries  

| Component | Model | Reasoning |
|-----------|-------|-----------|
| **Severity \(s_i\)** | Hierarchical Bayesian regression using sensor vitals, caller description, location risk factors. | Gives calibrated posterior variance → feeds directly into risk‑adjusted ELS. |
| **False‑call probability \(p_i\)** | **Adversarial Bayesian network**: prior probability of a false call \(\pi_{\text{adv}}\) (learned from historic false‑call rates) multiplied by features that indicate “gaming” (e.g., repeated calls from same phone, anomalous temporal patterns, known hot‑spot of previous abuse). Updated online with a *particle filter* to capture rapid changes. | Explicitly separates natural noise from strategic deception. |
| **Travel‑time \(\tau^{\text{arrival}}_i\)** | Stochastic forecast (Gaussian Process) conditioned on weather, air‑traffic, and helicopter state. | Captures sensor failure (e.g., GPS loss) by inflating variance. |
| **Adversary game** | **Stackelberg security game**: the system (leader) commits to a dispatch policy; the adversary (follower) chooses call locations to maximise expected wasted dispatches. The optimal policy is the *min‑max* (or *risk‑adjusted CVaR*) solution of the MILP. | Guarantees robustness against strategic false‑calls without needing to know the exact attacker type. |

### 3. Algorithmic details  

* **Robust MILP** formulation (variables \(x_{i,t}\), auxiliary worst‑case variables \(z\)):  

\[
\begin{aligned}
\max_{x,z} \; & \min_{s\in\mathcal{S}} \; \Bigl[ \sum_i \bigl( r(s_i^{(s)}) (1-p_i^{(s)})\bigr) x_{i,t} \\
& \qquad - w_T\sum_i \text{TTP}_i^{(s)}x_{i,t} - w_E\sum_d\text{EP}_d^{(s)} - w_A\sum_i\lambda_{\text{adv}}p_i^{(s)}x_{i,t} \Bigr] \\
\text{s.t. } & \text{capacity constraints (identical for every scenario)}\\
& z \le \text{objective of each scenario } s \\
& \text{explainability rule (confidence }>\theta_{\text{exp}})
\end{aligned}
\]

* **Solution method** – use a *Bender’s decomposition* where the master problem decides \(x\) and sub‑problems evaluate each scenario. In practice, with ≤ 30 simultaneous incidents (typical for a mid‑size city) the MILP solves in < 250 ms on a modest‑size CPU (e.g., Intel Xeon E5‑2670).  

* **Latency budget** – the pipeline is parallelised: sensor processing on GPU (≈ 30 ms), scenario generation on a separate thread, MILP on a dedicated optimisation core. The total wall‑clock time is bounded at **≤ 0.6 s**.

### 4. Deployment considerations  

| Aspect | Decision | Rationale |
|--------|----------|-----------|
| **Hardware** | Redundant edge‑servers (2 × 8‑core CPU, 32 GB RAM) co‑located at the dispatch centre; hot‑standby cloud fallback for peak loads. | Guarantees operation despite local hardware failure; latency budget preserved because the MILP is small enough to run locally. |
| **Software stack** | Python‑based data ingestion (FastAPI), NumPy/Scipy for Bayesian updates, Gurobi/CPLEX (commercial solver) for MILP, Dockerised micro‑services. | Proven libraries, easy to audit, and containerisation satisfies explainability and reproducibility requirements. |
| **Fail‑safe** | If optimisation exceeds 800 ms or any sensor stream is lost, a **rule‑based heuristic** (nearest‑available‑helicopter, severity‑threshold triage) is executed. | Guarantees that a decision is always made; the heuristic is fully auditable and can be overridden manually. |
| **Explainability interface** | Every dispatch record stores: (i) input features, (ii) posterior severity & false‑call probabilities, (iii) contribution of each objective term to the final score. A concise human‑readable “decision card” is auto‑generated for oversight committees. | Meets legal transparency obligations while keeping the model internals (weights, Bayesian priors) proprietary if required. |

---

## (C)  Data Pipeline & Privacy‑Preserving Methods  

### 1. Data sources (raw)  

| Source | Raw fields (examples) | Frequency |
|--------|-----------------------|-----------|
| **

## For the next cell, we will use Ollama

Ollama runs a local web service that gives an OpenAI compatible endpoint,  
and runs models locally using high performance C++ code.

If you don't have Ollama, install it here by visiting https://ollama.com then pressing Download and following the instructions.

After it's installed, you should be able to visit here: http://localhost:11434 and see the message "Ollama is running"

You might need to restart Cursor (and maybe reboot). Then open a Terminal (control+\`) and run `ollama serve`

Useful Ollama commands (run these in the terminal, or with an exclamation mark in this notebook):

`ollama pull <model_name>` downloads a model locally  
`ollama ls` lists all the models you've downloaded  
`ollama rm <model_name>` deletes the specified model from your downloads

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Super important - ignore me at your peril!</h2>
            <span style="color:#ff7800;">The model called <b>llama3.3</b> is FAR too large for home computers - it's not intended for personal computing and will consume all your resources! Stick with the nicely sized <b>llama3.2</b> or <b>llama3.2:1b</b> and if you want larger, try llama3.1 or smaller variants of Qwen, Gemma, Phi or DeepSeek. See the <A href="https://ollama.com/models">the Ollama models page</a> for a full list of models and sizes.
            </span>
        </td>
    </tr>
</table>

In [12]:
!ollama pull llama3.2

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 1.1 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 2.0 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 5.2 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 7.9 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 9.5 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  11 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  14 MB/2.0 GB                  pulling manifest 
pulling dd

In [13]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "llama3.2"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

I'll address each part of your question:

(A) **Precise Formal Objective Function** 

Given the constraints and objectives, I propose a multi-objective formulation with the following components:

1.  **Expected Lives Saved (ELM):**

    *   This objective aims to maximize survival probability while also considering the resource ceiling.

    \[E[L] = E[T] - w_{\text{resource}} E[R] \]

    where:

*   \(T\) represents the total time available within the fixed budget,
*   \(R\) is a measure of resource usage (number of helicopters deployed),
*   and $w_\mathrm{resource}$ is an adjustment factor depending on the availability of resources.



2.  **Time-to-Treatment for High-Risk Patients (TTTHRP):**

    \[TT_{HR} = E[T] - \omega^{\text{HTT}}E[R]^H\]

    where \( \omega^\mathrm{HTT}\) is a weighting factor adjusting the dependency on resource usage when evaluating treatments,
    and $E\text [R]$ is the number of available resources.



3.  **Equitable Service across Demographic Groups (ESDG):**

    \[E[SG] = \gamma^{\text{ESD}}S_\mathrm{eq} G^\mathrm{d}\]

    where \( S_\mathrm{eq}\) represents equity in service and $G^\mathrm{d}$ represents demographic diversity.

4.  **Robustness to Gaming (RG):**

    \[E[R] = \pi^{\text{RG}}R^\prime\\]

    Here, \( R^\prime\) represents an adjusted measure of resources that account for game strategies and potential underreporting, with \( \pi^\mathrm{RG} \) being a robustness adjustment factor.



(B) **Algorithmic Approach**

To address challenges like false calls and uncertainty, we integrate adversarial modeling into our dispatch algorithm. Here's a high-level overview:

*   First, we collect metadata on previously dispatched helicopters. A meta-data set helps create a predictive model of realistic resource demand under given situations.
*   We train machine learning models to predict the likelihood of false emergencies based on historical trends and call patterns. These predictions are fed into our decision-making algorithm.
*   The dispatcher uses probability outputs from the trained models, combined with other factors like time elapsed since last emergency and call urgency signs, to decide whether a resource can be allocated or not.

We will use techniques such as reinforcement learning for robustness in game scenarios and probabilistic inference for handling uncertainty.

(C) **Data Pipeline**

1.  **Sensors**: We integrate real-time sensor feeds, focusing on parameters like location, speed of movement, vital signs, etc.
2.  **Call Metadata**: Collect aggregated data on call patterns (e.g., frequency, distribution of locations), usage trends in resource usage based on helicopter movements, response time before emergency services arrive, and so on.

3.  To ensure privacy while aggregating raw sensor inputs without directly accessing personal health records, we use techniques such as:

*   Anonymization methods (like pseudonomization and location-based anonymization) to obscure individual identities and linkable data within the feed
*   Noise insertion techniques in raw sensor feeds to further prevent direct inference into personal records

(D) **Evaluation Plan**

1.  **Pre-Deployment Validation**: Conduct both internal analysis and external evaluations using established metrics including but not limited to:

    *   Lives saved (L),
    \[  \frac{EL}{ER} = \frac{\overline{T}}
        - w_{\textresource} \overline{R}} {}
    where \\
        \\   ER
            is the current baseline mortality rate,
        \\
          \text{LR}


    *   Time to treatment (TT)
    \[ TT=\overline{T^i}-\\omega\times
         {T} ]
    -where
    $
    T
        i$ represents time when response starts. In our case:
    $
\  E[\pi_i T]$

    *   Equitability score, measuring services performed across different demographic demographics (
    \[ E[S]
            G^d]


The plan should include continuous monitoring and adjustments as needed.


(E) **Anticipated Failure Modes

1.  Technical failures: We anticipate model drift due to changes in call patterns or sensor technologies.

2.  Behavioral failures: This might involve intentional manipulation (game-playing), miscommunication in public, etc.

3.  Social failures:

    \--

Assessing failure, mitigating strategies include:

4.  Model validation and learning via online feedback
5.  Robustness testing against known manipulations or adversarial models
6.  Integration of human oversight for critical decision-making to prevent misuse



(F) **Clarifying Questions**

1.  What are the exact regions with greatest needs, health disparities, etc., that our system should focus on?
2.  Are there any legal/privacy regulations we need to comply with as the public service system grows?

To fully address these questions, city officials can clarify their priorities and specific challenges faced by using systems like ours. Their input will significantly enhance the effectiveness and ethical stance of the Automated Decision Making tool designed for emergency services in your city.

In [14]:
# So where are we?

print(competitors)
print(answers)


['gpt-5-nano', 'claude-sonnet-4-5', 'gemini-2.5-flash', 'deepseek-chat', 'openai/gpt-oss-120b', 'llama3.2']
['Below is a concrete design sketch for an automated decision-making system (ADS) to dispatch a fixed fleet of emergency medical helicopters under the constraints you stated. I state key assumptions explicitly, then present (A)–(F) as requested, with trade-offs justified.\n\nOverview and core philosophy\n- We operate in a real-time, noisy-data setting with adversarial risk (false emergency calls) and significant demographic disparities. Raw PII is off-limits; we rely on noisy sensor streams, call metadata, and aggregated historical outcomes.\n- The objective is multi-objective and risk-aware: maximize expected lives saved (primary), minimize time-to-treatment for high-risk patients, ensure equitable access across demographic groups, and be robust to adversarial gaming and sensor failures, all within a fixed budget and with public explainability requirements.\n- We implement a rol

In [15]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


Competitor: gpt-5-nano

Below is a concrete design sketch for an automated decision-making system (ADS) to dispatch a fixed fleet of emergency medical helicopters under the constraints you stated. I state key assumptions explicitly, then present (A)–(F) as requested, with trade-offs justified.

Overview and core philosophy
- We operate in a real-time, noisy-data setting with adversarial risk (false emergency calls) and significant demographic disparities. Raw PII is off-limits; we rely on noisy sensor streams, call metadata, and aggregated historical outcomes.
- The objective is multi-objective and risk-aware: maximize expected lives saved (primary), minimize time-to-treatment for high-risk patients, ensure equitable access across demographic groups, and be robust to adversarial gaming and sensor failures, all within a fixed budget and with public explainability requirements.
- We implement a rolling-horizon, robust, multi-objective dispatch policy with a human-in-the-loop fallback whe

In [16]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [17]:
print(together)

# Response from competitor 1

Below is a concrete design sketch for an automated decision-making system (ADS) to dispatch a fixed fleet of emergency medical helicopters under the constraints you stated. I state key assumptions explicitly, then present (A)–(F) as requested, with trade-offs justified.

Overview and core philosophy
- We operate in a real-time, noisy-data setting with adversarial risk (false emergency calls) and significant demographic disparities. Raw PII is off-limits; we rely on noisy sensor streams, call metadata, and aggregated historical outcomes.
- The objective is multi-objective and risk-aware: maximize expected lives saved (primary), minimize time-to-treatment for high-risk patients, ensure equitable access across demographic groups, and be robust to adversarial gaming and sensor failures, all within a fixed budget and with public explainability requirements.
- We implement a rolling-horizon, robust, multi-objective dispatch policy with a human-in-the-loop fallba

In [18]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [19]:
print(judge)

You are judging a competition between 6 competitors.
Each model has been given this question:

You are responsible for designing an automated decision-making system (ADS) to allocate a limited fleet of emergency medical helicopters in a mid-sized city that has significant demographic disparities and a risk of adversarial actors who may place false emergency calls to monopolize resources. Raw personal medical records are off-limits; you only have access to noisy real-time sensor feeds, call metadata, and historical anonymized aggregated outcomes. Your objectives, in descending priority, are: (1) maximize lives saved (measured as expected reduction in mortality versus the current baseline), (2) minimize time-to-treatment for high-risk patients, (3) ensure equitable service across demographic groups, and (4) be robust to gaming and sensor failure—while operating under a fixed budget and legal/explainability constraints for public oversight. Describe the following: (A) a precise formal obj

In [20]:
judge_messages = [{"role": "user", "content": judge}]

In [21]:
# Judgement time!

openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)


{"results": ["1", "3", "4", "5", "2", "6"]}


In [22]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

Rank 1: gpt-5-nano
Rank 2: gemini-2.5-flash
Rank 3: deepseek-chat
Rank 4: openai/gpt-oss-120b
Rank 5: claude-sonnet-4-5
Rank 6: llama3.2


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>